In [1]:
import warnings
warnings.filterwarnings('ignore')
import glob
from sigpyproc.readers import FilReader
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib
from matplotlib import dates
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator
from datetime import datetime, timedelta
import astropy.units as u
from astropy.time import Time
from astropy.visualization import ImageNormalize, PercentileInterval
from astropy.io import fits as pyfits
import matplotlib as mpl
from sunpy.net import attrs as sunpy_attrs
from radiospectra.spectrogram import Spectrogram

mpl.rcParams['date.epoch'] = '1970-01-01T00:00:00' # use precise epoch
try:
    mdates.set_epoch('1970-01-01T00:00:00')
except:
    pass

# plt.rcParams['figure.dpi'] = 100
# plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

path = '/home/mnedal/data/I-LOFAR'
stokes = 'I'
mydatetime = '2025-10-06T08:50:00'

In [2]:
def get_ilofar_file(paths, datetime):
    """
    Find the full path of the I-LOFAR file 
    with a given datetime in a list of paths.
    datetime -> YYYY-mm-ddTHH:MM:SS
    """
    for p in paths:
        parts = p.split('_')
        try:
            dt = p.split('_')[1]
        except ValueError:
            continue
        if datetime == dt:
            return p
    return None

In [3]:
def freq_axis(freqs):
    """
    Introduce gaps in the frequency axis of I-LOFAR REALTA data.
    """
    gap1 = np.flipud(freqs[288]+(np.arange(59)*0.390625))
    gap2 = np.flipud(freqs[88]+(np.arange(57)*0.390625))
    ax_shape = 59+57-1
    new_freq = np.zeros(ax_shape+freqs.shape[0])
    
    new_freq[0:88] = freqs[0:88]
    new_freq[88:145]  = gap2[:57]
    new_freq[145:345] = freqs[88:288]
    new_freq[345:404] = gap1[:59]
    new_freq[404:] = freqs[289:]
    
    return new_freq

In [4]:
def df_limits(df):
    vals = df.values.ravel()
    vmin = np.nanpercentile(vals, 5)
    vmax = np.nanpercentile(vals, 99.7)
    return vmin, vmax

In [5]:
def spec_limits(radiospec):
    vals = radiospec.data.ravel()
    vmin = np.nanpercentile(vals, 5)
    vmax = np.nanpercentile(vals, 98.7)
    return vmin, vmax

In [6]:
ilofar_paths = sorted(glob.glob(f'{path}/*.fil'))
print(*ilofar_paths, sep='\n')

/home/mnedal/data/I-LOFAR/Sun357_20240514_stokesI.fil
/home/mnedal/data/I-LOFAR/Sun357_20240514_stokesV.fil
/home/mnedal/data/I-LOFAR/Sun357_2025-03-25_typeII.fil
/home/mnedal/data/I-LOFAR/Sun357_2025-03-26T08:31:00_StokesI_dyspec.fil
/home/mnedal/data/I-LOFAR/Sun357_2025-03-26_typeII_1.fil
/home/mnedal/data/I-LOFAR/Sun357_2025-03-26_typeII_2.fil
/home/mnedal/data/I-LOFAR/Sun357_2025-10-06T08:50:00_StokesI_dyspec.fil
/home/mnedal/data/I-LOFAR/realta_ilofar_stokesI_20240514T1635.fil
/home/mnedal/data/I-LOFAR/realta_ilofar_stokesI_20240514T1635_masked.fil


In [7]:
filename = get_ilofar_file(ilofar_paths, mydatetime)
print(filename)

a = FilReader(filename) # header
header = a.header.to_dict()

/home/mnedal/data/I-LOFAR/Sun357_2025-10-06T08:50:00_StokesI_dyspec.fil


In [ ]:
tstart_obs_str = Time(a.header.tstart, format='mjd').iso
n_samples      = a.header.nsamples
print(tstart_obs_str, n_samples, sep='\n')

data = a.read_block(start=0, nsamps=n_samples)
print(data.shape)

2025-10-06 08:49:59.996
3204346


In [ ]:
# making time axis
tstart = Time(data.header.tstart, format='mjd')                    # tstart.iso will tell the time in format yyyy-mm-dd hh:mm:ss
tarray = tstart + (np.arange(data.shape[1])*data.header.tsamp*u.s) # making the time array for realta time resolution
print(len(tarray), tarray[0].iso, tarray[-1].iso, sep='\n')

In [ ]:
dt = datetime.strptime(tarray[1].iso, '%Y-%m-%d %H:%M:%S.%f') - datetime.strptime(tarray[0].iso, '%Y-%m-%d %H:%M:%S.%f')
print('Time cadence:', dt.total_seconds()*1000, 'ms.')

In [ ]:
# Converting the array to datetime object
Tarray = [datetime.strptime(t.iso, '%Y-%m-%d %H:%M:%S.%f') for t in tarray]
print(Tarray[0], Tarray[-1], sep='\n')

In [ ]:
# export the frequency axis
freqs = data.header.chan_freqs
print(freqs[0], freqs[-1], sep='\n')

In [ ]:
new_freq = freq_axis(freqs)

data = np.log10(data)
data[np.where(np.isinf(data)==True)] = 0.0

data2 = np.empty((new_freq.shape[0], data.shape[1]))    
data2[:] = np.NaN
data2[0:88] = data[0:88]
data2[145:345] = data[88:288]
data2[404:] = data[289:]

In [ ]:
freq_mode3 = np.linspace(10, 90, 199)
freq_mode5 = np.linspace(110, 190, 200)
freq_mode7 = np.linspace(210, 270, 88)

df_mode3 = pd.DataFrame(data=data2[404:].T, columns=freq_mode3[::-1])
df_mode5 = pd.DataFrame(data=data2[145:345].T, columns=freq_mode5[::-1])
df_mode7 = pd.DataFrame(data=data2[:88].T, columns=freq_mode7[::-1])

In [ ]:
print(df_mode3.shape, df_mode5.shape, df_mode7.shape, sep='\n')

In [ ]:
# Ensure the 'time' column is in datetime format
df_mode3.index = pd.to_datetime(Tarray)
df_mode5.index = pd.to_datetime(Tarray)
df_mode7.index = pd.to_datetime(Tarray)

In [ ]:
# Save the dataframes as a pickle files
df_mode3.to_pickle(f'{path}/df_mode3_realta_{mydatetime}_S{stokes}.pkl')
df_mode5.to_pickle(f'{path}/df_mode5_realta_{mydatetime}_S{stokes}.pkl')
df_mode7.to_pickle(f'{path}/df_mode7_realta_{mydatetime}_S{stokes}.pkl')

---

## Load the dataframes from the pickle files

In [ ]:
mydatetime = '20251006'
df_mode3 = pd.read_pickle(f'{path}/df_mode3_realta_{mydatetime}_S{stokes}.pkl')
df_mode5 = pd.read_pickle(f'{path}/df_mode5_realta_{mydatetime}_S{stokes}.pkl')
df_mode7 = pd.read_pickle(f'{path}/df_mode7_realta_{mydatetime}_S{stokes}.pkl')

time_mode3 = df_mode3.index
time_mode5 = df_mode5.index
time_mode7 = df_mode7.index

freq_mode3 = df_mode3.columns
freq_mode5 = df_mode5.columns
freq_mode7 = df_mode7.columns

In [ ]:
df_mode3.head()

In [ ]:
# Slice the DataFrame between start_date and end_date
start_date = '2025-10-06 08:55:00'
end_date   = '2025-10-06 09:05:00'

df_mode3_slice = df_mode3.loc[start_date:end_date]
df_mode5_slice = df_mode5.loc[start_date:end_date]
df_mode7_slice = df_mode7.loc[start_date:end_date]

del df_mode3
del df_mode5
del df_mode7

## Resampling to the same cadence of ORFEES

In [ ]:
# Downsample to 1-second resolution for faster testing and visualization
df_mode3_100ms = df_mode3_slice.resample('100ms').mean()
df_mode5_100ms = df_mode5_slice.resample('100ms').mean()
df_mode7_100ms = df_mode7_slice.resample('100ms').mean()

In [ ]:
# remove the const background
mode3_new = df_mode3_100ms.values - np.tile(np.nanmean(df_mode3_100ms.values,0), (df_mode3_100ms.values.shape[0],1))
mode5_new = df_mode5_100ms.values - np.tile(np.nanmean(df_mode5_100ms.values,0), (df_mode5_100ms.values.shape[0],1))
mode7_new = df_mode7_100ms.values - np.tile(np.nanmean(df_mode7_100ms.values,0), (df_mode7_100ms.values.shape[0],1))

# construct dataframes again
df_mode3_new = pd.DataFrame(data=mode3_new, columns=freq_mode3)
df_mode5_new = pd.DataFrame(data=mode5_new, columns=freq_mode5)
df_mode7_new = pd.DataFrame(data=mode7_new, columns=freq_mode7)

# set the time column as the dataframe index
df_mode3_new.index = pd.to_datetime(df_mode3_100ms.index)
df_mode5_new.index = pd.to_datetime(df_mode5_100ms.index)
df_mode7_new.index = pd.to_datetime(df_mode7_100ms.index)

del mode3_new
del mode5_new
del mode7_new

In [ ]:
use_limits = True

fig, ax = plt.subplots(figsize=[12,5])

for df in [df_mode5_new, df_mode7_new]:
    if use_limits:
        vmin, vmax = df_limits(df)
        ax.pcolormesh(
            df.index,
            df.columns,
            df.values.T,
            cmap='RdYlBu_r',
            vmin=vmin,
            vmax=vmax
        )
    else:
        ax.pcolormesh(
            df.index,
            df.columns,
            df.values.T,
            cmap='RdYlBu_r'
        )
ax.xaxis.set_minor_locator(AutoMinorLocator(n=3))
ax.yaxis.set_minor_locator(AutoMinorLocator(n=4))
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlim(pd.Timestamp('2025-10-06T08:57'), pd.Timestamp('2025-10-06T09:02'))
ax.set_ylim(df_mode7_new.columns[0], df_mode5_new.columns[-1])
fig.tight_layout()
plt.show()

## Plot ORFEES

In [ ]:
data_dir = '/home/mnedal/data/ORFEES'
mydate = '2025-10-06'
year, month, day = mydate.split('-')
files = sorted(glob.glob(f'{data_dir}/*{year}{month}{day}_*.fts'))
print(*files, sep='\n')

orfees = pyfits.open(files[0])
orfees_i = np.hstack([orfees[2].data[f'STOKESI_B{i}'] for i in range(1, 6)]).T
data = orfees_i.T
print(data.shape)

# Remove the background by taking the data from the quiet background and divide it by the data
new_data = data - np.tile(np.nanmean(data,0), (data.shape[0],1))

new_data = new_data.T
orfees_time_str = orfees[0].header['DATE-OBS']
orfees_times = Time(orfees_time_str) + (orfees[2].data['TIME_B1']/1000)*u.s # times are not the same for all sub spectra
orfees_freqs = np.hstack([orfees[1].data[f'FREQ_B{i}'] for i in range(1, 6)])*u.MHz

orfees_meta = {
    'observatory': orfees[0].header['ORIGIN'],
    'instrument': orfees[0].header['INSTRUME'],
    'detector': orfees[0].header['INSTRUME'],
    'freqs': orfees_freqs.reshape(-1),
    'times': orfees_times,
    'wavelength': sunpy_attrs.Wavelength(orfees_freqs[0,0],
                                         orfees_freqs[0,-1]),
    'start_time': orfees_times[0],
    'end_time': orfees_times[-1]
}

orfees_spec_i = Spectrogram(new_data, orfees_meta)

In [ ]:
use_limits = True

fig, ax = plt.subplots(figsize=[12,5])

times = orfees_spec_i.times.datetime
freqs = orfees_spec_i.frequencies.value
data  = orfees_spec_i.data

if use_limits:
    vmin, vmax = spec_limits(orfees_spec_i)
    ax.pcolormesh(times, freqs, data, cmap='RdYlBu_r', vmin=vmin, vmax=vmax)
else:
    ax.pcolormesh(times, freqs, data, cmap='RdYlBu_r')

ax.xaxis.set_minor_locator(AutoMinorLocator(n=3))
ax.yaxis.set_minor_locator(AutoMinorLocator(n=4))
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlim(pd.Timestamp('2025-10-06T08:57'), pd.Timestamp('2025-10-06T09:02'))
ax.set_ylim(bottom=300, top=orfees_freqs.reshape(-1)[0].value)
fig.tight_layout()
plt.show()

## Plot I-LOFAR with ORFEES

In [ ]:
use_limits = True

fig, ax = plt.subplots(figsize=[10,5])

for df in [df_mode5_new, df_mode7_new]:
    if use_limits:
        ax.pcolormesh(
            df.index,
            df.columns,
            df.values.T,
            cmap='RdYlBu_r',
            vmin=df_limits(df)[0],
            vmax=df_limits(df)[1]
        )
        ax.pcolormesh(times, freqs, data, cmap='RdYlBu_r',
                      vmin=spec_limits(orfees_spec_i)[0],
                      vmax=spec_limits(orfees_spec_i)[1])
    else:
        ax.pcolormesh(
            df.index,
            df.columns,
            df.values.T,
            cmap='RdYlBu_r'
        )
        ax.pcolormesh(times, freqs, data, cmap='RdYlBu_r')

ax.grid(visible=True, which='both', color='k', alpha=1)
ax.xaxis.set_minor_locator(AutoMinorLocator(n=3))
ax.yaxis.set_minor_locator(AutoMinorLocator(n=4))
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlim(pd.Timestamp('2025-10-06T08:57'), pd.Timestamp('2025-10-06T09:02'))
# ax.set_ylim(bottom=450, top=df_mode5_new.columns[-1])
ax.set_ylim(bottom=280, top=df_mode5_new.columns[-1])
fig.tight_layout()
# fig.savefig('./ilofar_orfees_dyspec.png', format='png', bbox_inches='tight')
plt.show()

---
## Type II burst: shock and coronal magnetic-field diagnostics

The cells below estimate the frequency drift rate, source height, shock speed,
density jump, Alfven Mach number, Alfven speed and coronal magnetic field of
the band-split type II seen in both I-LOFAR and ORFEES on 2025-10-06
(~270-130 MHz, ~08:57:30-09:01:30 UT).

**Method** (Smerd et al. 1974; Vrsnak et al. 2002, 2004; as applied to
band-split type IIs by Gopalswamy et al. 2012; Vasanth et al. 2014). With the
harmonic number $s$ ($s=1$ fundamental, $s=2$ harmonic):

$$N_e = \left(\frac{f_{\rm obs}/s}{8.9787\times10^{3}}\right)^2\,{\rm cm^{-3}},
\qquad X \equiv \frac{N_{\rm down}}{N_{\rm up}} = \left(\frac{f_U}{f_L}\right)^2 .$$

$X$ is independent of $s$. For a low-$\beta$ perpendicular shock with
$\gamma=5/3$ the Rankine-Hugoniot relations give

$$M_A^2 = \frac{X(X+5)}{2(4-X)} ,$$

and, with the shock speed from the height-time gradient of the upstream
(lower) band,

$$v_A = \frac{v_{\rm sh}}{M_A}, \qquad B = v_A\sqrt{\mu_0\,\rho},
\qquad \rho = \mu\,m_p N_e .$$

**Conventions used here** (edit if your reading of the spectrum differs):
the *lower* band is the upstream/ambient plasma, so the height, $N_e$ and
$\rho$ are all taken from $f_L$; the upper band enters only through $X$.
The mean mass per electron defaults to $\mu=1.27$; a strict 10% He corona
gives $\mu\approx1.17$. Both harmonics are computed because the F/H identity
of the observed band is unknown.

In [ ]:
# Density-model helpers. These mirror coronal_density_models.ipynb; if you
# factor that notebook's functions into coronal_density_models.py the import
# below will pick them up, otherwise the inline definitions are used.
from scipy.optimize import brentq
from scipy.constants import m_p, mu_0

R_SUN_M      = 6.957e8       # solar radius [m]
PLASMA_CONST = 8.9787e3      # f_p [Hz] = PLASMA_CONST * sqrt(N_e [cm^-3])

try:
    from coronal_density_models import (freq_to_density, freq_to_radius,
                                        newkirk, saito, leblanc, mann2023,
                                        alfven_mach_from_X)
except Exception:
    def freq_to_density(f_obs_hz, harmonic=1):
        return (np.asarray(f_obs_hz, float) / harmonic / PLASMA_CONST) ** 2

    def newkirk(r, fold=1):
        return fold * 4.2e4 * 10.0 ** (4.32 / np.asarray(r, float))

    def saito(r, region="equatorial"):
        r = np.asarray(r, float)
        a1, b1, a2, b2 = (1.36e6, 2.14, 1.68e8, 6.13) if region == "equatorial" \
                          else (2.5e5, 2.5, 1.0e8, 6.0)
        return a1 * r ** (-b1) + a2 * r ** (-b2)

    def leblanc(r):
        r = np.asarray(r, float)
        return 3.3e5 / r ** 2 + 4.1e6 / r ** 4 + 8.0e7 / r ** 6

    def mann2023(r):
        table_r = np.array([1.0,1.1,1.2,1.3,1.4,1.5,1.6,1.8,2.0,2.5,3.0,4.0,5.0,
            6.0,8.0,10.0,12.0,14.0,16.0,20.0,25.0,30.0,40.0,50.0,60.0,70.0,80.0,
            100.0,120.0,160.0,200.0,215.0,250.0])
        table_ne = np.array([7.17e8,2.60e8,1.12e8,5.46e7,2.96e7,1.74e7,1.09e7,
            5.02e6,2.69e6,8.58e5,3.89e5,1.33e5,6.42e4,3.70e4,1.65e4,9.10e3,5.70e3,
            3.88e3,2.80e3,1.63e3,9.66e2,6.32e2,3.28e2,1.98e2,1.32e2,9.37e1,6.98e1,
            4.28e1,2.83e1,1.54e1,9.53,8.16,5.91])
        r = np.asarray(r, float)
        log_r = np.log10(np.clip(r, table_r.min(), table_r.max()))
        return 10.0 ** np.interp(log_r, np.log10(table_r), np.log10(table_ne))

    def freq_to_radius(f_obs_hz, model, harmonic=1, r_bounds=(1.001, 250.0),
                       **kw):
        f_obs = np.atleast_1d(np.asarray(f_obs_hz, float))
        ne_t = freq_to_density(f_obs, harmonic=harmonic)
        r_lo, r_hi = r_bounds
        out = np.empty_like(f_obs)
        for i, nt in enumerate(ne_t):
            ne_lo, ne_hi = model(r_lo, **kw), model(r_hi, **kw)
            if not (min(ne_lo, ne_hi) <= nt <= max(ne_lo, ne_hi)):
                out[i] = np.nan
                continue
            try:
                out[i] = brentq(lambda r: model(r, **kw) - nt, r_lo, r_hi)
            except ValueError:
                out[i] = np.nan
        return out if out.size > 1 else float(out[0])

    def alfven_mach_from_X(X, gamma=5.0 / 3.0):
        X = np.asarray(X, float)
        return np.sqrt(X * (X + 5.0) / (2.0 * (4.0 - X)))

In [ ]:
# Build (times, freqs, data) arrays for the picker. ORFEES is continuous over
# 144-1000 MHz so it is the cleaner panel to click on; the I-LOFAR block below
# is provided if you want to extend the picks below ~144 MHz.

# --- ORFEES ---
orf_times = pd.to_datetime(orfees_spec_i.times.datetime)
orf_freqs = orfees_spec_i.frequencies.value          # MHz
orf_data  = orfees_spec_i.data                        # shape (n_freq, n_time)

# --- I-LOFAR (mode5 110-190 + mode7 210-270, already background-subtracted) ---
ilofar_times = df_mode5_new.index
ilofar_freqs = np.concatenate([df_mode7_new.columns.values,
                               df_mode5_new.columns.values])      # MHz
ilofar_data  = np.vstack([df_mode7_new.values.T, df_mode5_new.values.T])

print("ORFEES  :", orf_data.shape, f"{orf_freqs.min():.0f}-{orf_freqs.max():.0f} MHz")
print("I-LOFAR :", ilofar_data.shape, f"{ilofar_freqs.min():.0f}-{ilofar_freqs.max():.0f} MHz")

In [ ]:
pick_style = 'manual'

if pick_style == 'auto':
    def pick_type2_lanes(times, freqs_mhz, data, fmin=130, fmax=280,
                         tmin="2025-10-06 08:57:00", tmax="2025-10-06 09:02:00",
                         percentiles=(5, 99)):
        """Interactive UF/LF lane picker for a dynamic spectrum (remote-safe).
    
        Click points along each lane; toggle which lane you are tracing. Picks are
        stored in the kernel and returned as a dict of lists. Uses plotly
        FigureWidget so clicks work in JupyterLab over SSH without ipympl.
        """
        import plotly.graph_objects as go
        import ipywidgets as widgets
        from IPython.display import display
    
        times = pd.to_datetime(np.asarray(times))
        freqs_mhz = np.asarray(freqs_mhz, float)
        fo = np.argsort(freqs_mhz)
        freqs_mhz, data = freqs_mhz[fo], np.asarray(data)[fo, :]
    
        fsel = (freqs_mhz >= fmin) & (freqs_mhz <= fmax)
        tsel = (times >= pd.Timestamp(tmin)) & (times <= pd.Timestamp(tmax))
        ff, tt = freqs_mhz[fsel], times[tsel]
        z = data[np.ix_(fsel, tsel)]
        zmin, zmax = np.nanpercentile(z, percentiles)
    
        picks = {"UF": {"t": [], "f": []}, "LF": {"t": [], "f": []}}
    
        fig = go.FigureWidget([
            go.Heatmap(x=tt, y=ff, z=z, colorscale="RdBu_r",
                       zmin=zmin, zmax=zmax, showscale=False),
            go.Scatter(x=[], y=[], mode="markers", name="UF (downstream)",
                       marker=dict(color="red", size=8, symbol="x")),
            go.Scatter(x=[], y=[], mode="markers", name="LF (upstream)",
                       marker=dict(color="deepskyblue", size=8, symbol="x")),
        ])
        fig.update_layout(height=520, width=950, xaxis_title="Time (UT)",
                          yaxis_title="Frequency (MHz)",
                          margin=dict(l=60, r=20, t=30, b=50),
                          legend=dict(orientation="h", y=1.08))
    
        lane = widgets.ToggleButtons(options=["UF (downstream)", "LF (upstream)"],
                                     value="UF (downstream)")
    
        def redraw(key):
            idx = 1 if key == "UF" else 2
            with fig.batch_update():
                fig.data[idx].x = picks[key]["t"]
                fig.data[idx].y = picks[key]["f"]
    
        def on_click(trace, points, state):
            if len(points.xs) == 0:
                return
            key = "UF" if lane.value.startswith("UF") else "LF"
            picks[key]["t"].append(pd.to_datetime(points.xs[0]))
            picks[key]["f"].append(float(points.ys[0]))
            redraw(key)
    
        fig.data[0].on_click(on_click)
    
        undo = widgets.Button(description="Undo last", icon="rotate-left")
        clr = widgets.Button(description="Clear lane", icon="trash")
    
        def _undo(_):
            key = "UF" if lane.value.startswith("UF") else "LF"
            for k in ("t", "f"):
                if picks[key][k]:
                    picks[key][k].pop()
            redraw(key)
    
        def _clr(_):
            key = "UF" if lane.value.startswith("UF") else "LF"
            picks[key]["t"].clear(); picks[key]["f"].clear()
            redraw(key)
    
        undo.on_click(_undo); clr.on_click(_clr)
        display(widgets.VBox([widgets.HBox([lane, undo, clr]), fig]))
        return picks
    
    
    # Click the upper band, toggle to LF, click the lower band. Re-run this cell
    # to start over. Switch the first argument set to the I-LOFAR arrays to pick
    # the <144 MHz extension.
    picks = pick_type2_lanes(orf_times, orf_freqs, orf_data)

elif pick_style == 'manual':
    picks = {
    'UF': {'t': list(pd.to_datetime([
        '2025-10-06 08:58:20', '2025-10-06 08:58:40', '2025-10-06 08:59:00',
        '2025-10-06 08:59:20', '2025-10-06 08:59:40', '2025-10-06 09:00:00',
        '2025-10-06 09:00:20', '2025-10-06 09:00:40', '2025-10-06 09:01:00'])),
           'f': [250, 240, 220, 210, 220, 210, 190, 180, 165]},      # upper band, MHz
    'LF': {'t': list(pd.to_datetime([
        '2025-10-06 08:58:20', '2025-10-06 08:58:40', '2025-10-06 08:59:00',
        '2025-10-06 08:59:20', '2025-10-06 08:59:40', '2025-10-06 09:00:00',
        '2025-10-06 09:00:20', '2025-10-06 09:00:40', '2025-10-06 09:01:00'])),
           'f': [215, 200, 185, 175, 180, 170, 160, 150, 135]},      # lower band, MHz
    }

In [ ]:
# Turn the clicks into smooth lane fits, then run the full diagnostic chain.

t0 = min(picks["UF"]["t"] + picks["LF"]["t"])

def _lane_arrays(key, deg=2):
    t = pd.to_datetime(picks[key]["t"]); f = np.asarray(picks[key]["f"], float)
    ts = (t - t0).total_seconds().to_numpy()
    o = np.argsort(ts)
    ts, f = ts[o], f[o]
    return ts, f, np.polyfit(ts, f, min(deg, len(ts) - 1))

tU, fU_pts, pU = _lane_arrays("UF")
tL, fL_pts, pL = _lane_arrays("LF")

# common time grid over the overlap of the two lanes
tg = np.linspace(max(tU.min(), tL.min()), min(tU.max(), tL.max()), 60)
fU = np.polyval(pU, tg)                       # MHz
fL = np.polyval(pL, tg)                       # MHz
driftU = np.polyval(np.polyder(pU), tg)       # MHz/s (negative = drift to low f)
driftL = np.polyval(np.polyder(pL), tg)

X = (fU / fL) ** 2                            # density jump (harmonic-free)
MA = alfven_mach_from_X(X)

mu = 1.27                                     # mean mass per electron; ~1.17 for 10% He

def shock_field(model, harmonic, deg=2):
    """Height, shock speed, N_up, v_A and B along the grid for one model+harmonic."""
    r = freq_to_radius(fL * 1e6, model, harmonic=harmonic)   # upstream band -> height
    good = np.isfinite(r)
    v_sh = np.full_like(tg, np.nan)
    if good.sum() > deg:
        pr = np.polyfit(tg[good], r[good], deg)
        v_sh = np.polyval(np.polyder(pr), tg) * R_SUN_M / 1e3      # km/s
    n_up = freq_to_density(fL * 1e6, harmonic=harmonic)            # cm^-3
    v_A = v_sh / MA                                               # km/s
    rho = mu * m_p * (n_up * 1e6)                                 # kg/m^3
    B = (v_A * 1e3) * np.sqrt(mu_0 * rho) * 1e4                   # Gauss
    return dict(r=r, v_sh=v_sh, n_up=n_up, v_A=v_A, B=B)

models = [("Newkirk x1", lambda r: newkirk(r, 1)),
          ("Newkirk x2", lambda r: newkirk(r, 2)),
          ("Newkirk x4", lambda r: newkirk(r, 4)),
          ("Saito eq",   lambda r: saito(r, "equatorial"))]

print(f"Drift rate (UF): {np.nanmean(driftU):+.3f} MHz/s   "
      f"(LF): {np.nanmean(driftL):+.3f} MHz/s")
print(f"Density jump X = {np.nanmean(X):.3f} "
      f"({np.nanmin(X):.3f}-{np.nanmax(X):.3f}),   "
      f"M_A = {np.nanmean(MA):.2f}\n")

hdr = f"{'Model':<11}{'s':<3}{'r [Rs]':<14}{'v_sh [km/s]':<14}" \
      f"{'v_A [km/s]':<13}{'B [G]':<14}"
print(hdr); print("-" * len(hdr))
results = {}
for name, model in models:
    for s in (1, 2):
        d = shock_field(model, s)
        results[(name, s)] = d
        rng = lambda a, f="{:.2f}": (f"{f.format(np.nanmin(a))}-{f.format(np.nanmax(a))}"
                                     if np.isfinite(a).any() else "  -  ")
        print(f"{name:<11}{s:<3}{rng(d['r']):<14}{rng(d['v_sh'],'{:.0f}'):<14}"
              f"{rng(d['v_A'],'{:.0f}'):<13}{rng(d['B']):<14}")

In [ ]:
# Summary: picked lanes (band-independent), then height-time and B(r) under
# BOTH the fundamental (s=1) and harmonic (s=2) assumptions.
ref_model = "Newkirk x2"                       # density model shown in panel (b)
harm_style = {1: dict(ls="-",  tag="F"),
              2: dict(ls="--", tag="H")}

fig, ax = plt.subplots(1, 3, figsize=[15,4])

# (a) picked lanes over the ORFEES spectrum -- independent of F/H
sel_t = (orf_times >= pd.Timestamp("2025-10-06 08:57:00")) & \
        (orf_times <= pd.Timestamp("2025-10-06 09:02:00"))
sel_f = (orf_freqs >= 120) & (orf_freqs <= 290)
fo = np.argsort(orf_freqs[sel_f])
vmin, vmax = np.nanpercentile(orf_data[np.ix_(sel_f, sel_t)], (5, 99))
ax[0].pcolormesh(orf_times[sel_t], orf_freqs[sel_f][fo],
                 orf_data[np.ix_(sel_f, sel_t)][fo], cmap="RdYlBu_r",
                 vmin=vmin, vmax=vmax)
ax[0].plot(t0 + pd.to_timedelta(tg, "s"), fU, color="red",  lw=1.5, label="UF fit")
ax[0].plot(t0 + pd.to_timedelta(tg, "s"), fL, color="navy", lw=1.5, label="LF fit")
ax[0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax[0].set_xlabel("Time (UT)")
ax[0].set_ylabel("Frequency (MHz)")
ax[0].set_title('Band-split lanes')
ax[0].legend(fontsize=8)
ax[0].set_ylim(bottom=145, top=280)

# (b) height-time and shock speed for ref_model, both harmonics
ax2 = ax[1]
axb = ax2.twinx()
for s, st in harm_style.items():
    d = results[(ref_model, s)]
    ax2.plot(tg, d["r"],    st["ls"], color="firebrick", label=f"height ({st['tag']})")
    axb.plot(tg, d["v_sh"], st["ls"], color="steelblue", label=f"$v_{{sh}}$ ({st['tag']})")
ax2.set_xlabel("time since onset [s]")
ax2.set_ylabel(r"$r\,/\,R_\odot$", color="firebrick")
axb.set_ylabel(r"$v_{\rm sh}$ [km/s]", color="steelblue")
ax2.set_title(f"Height-time  ({ref_model})")
ax2.legend(loc="upper left", fontsize=8)

# (c) B(r) for all models, both harmonics: colour = model, line style = F/H
colors = plt.cm.viridis(np.linspace(0, 0.85, len(models)))
ax3 = ax[2]
for (name, _), c in zip(models, colors):
    for s, st in harm_style.items():
        dd = results[(name, s)]
        if np.isfinite(dd["r"]).any():
            ax3.plot(dd["r"], dd["B"], st["ls"], color=c, lw=1.4)
rr = np.linspace(1.05, 2.0, 100)
mag_field = 0.5 * (rr - 1.0) ** (-1.5)
ax3.plot(rr, mag_field, "k:", lw=1)        # Dulk-McLean 1978
ax3.set_xlabel(r"$r\,/\,R_\odot$"); ax3.set_ylabel(r"$B$ [G]")
ax3.set_yscale("log"); ax3.set_title("Coronal magnetic field")

from matplotlib.lines import Line2D
model_handles = [Line2D([], [], color=c, lw=2, label=n)
                 for (n, _), c in zip(models, colors)]
band_handles = [Line2D([], [], color="0.3", ls="-",  label="fundamental"),
                Line2D([], [], color="0.3", ls="--", label="harmonic"),
                Line2D([], [], color="k",   ls=":",  label="Dulk-McLean 78")]
leg1 = ax3.legend(handles=model_handles, fontsize=7, loc="upper right")
ax3.add_artist(leg1)
ax3.legend(handles=band_handles, fontsize=7, loc="lower left")

fig.tight_layout()
# fig.savefig(f'./dyspec_summary.png', format='png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=[8,3])

# (a) picked lanes over the ORFEES spectrum -- independent of F/H
ax[0].pcolormesh(orf_times[sel_t], orf_freqs[sel_f][fo],
                 orf_data[np.ix_(sel_f, sel_t)][fo], cmap="RdYlBu_r",
                 vmin=vmin, vmax=vmax)
times_t0 = t0 + pd.to_timedelta(tg, 's')
ax[0].plot(times_t0, fU, color="red",  lw=1.5, label="UF fit")
ax[0].plot(times_t0, fL, color="navy", lw=1.5, label="LF fit")
ax[0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax[0].set_xlabel("Time (UT)")
ax[0].set_ylabel("Frequency (MHz)")
ax[0].set_title('Band-split lanes')
ax[0].legend(fontsize=8)
ax[0].set_ylim(bottom=145, top=280)

# (b) height-time and shock speed for ref_model, both harmonics
ax2 = ax[1]
axb = ax2.twinx()
for s, st in harm_style.items():
    d = results[(ref_model, s)]
    ax2.plot(tg, d["r"],    st["ls"], color="firebrick", label=f"height ({st['tag']})")
    axb.plot(tg, d["v_sh"], st["ls"], color="steelblue", label=f"$v_{{sh}}$ ({st['tag']})")
ax2.set_xlabel("time since onset [s]")
ax2.set_ylabel(r"$r\,/\,R_\odot$", color="firebrick")
axb.set_ylabel(r"$v_{\rm sh}$ [km/s]", color="steelblue")
ax2.set_title(f"Height-time  ({ref_model})")
ax2.legend(loc="upper left", fontsize=8)

fig.tight_layout()
fig.savefig(f'./dyspec_summary.png', format='png', bbox_inches='tight')
plt.show()

In [ ]:
# # Summary: I-LOFAR + ORFEES spectrum with picked lanes, then height-time and
# # B(r) under BOTH the fundamental (s=1) and harmonic (s=2) assumptions.
# ref_model = "Newkirk x2"                       # density model shown in panel (b)
# harm_style = {1: dict(ls="-",  tag="F"),
#               2: dict(ls="--", tag="H")}

# fig, ax = plt.subplots(1, 3, figsize=(15, 5))

# # (a) I-LOFAR (mode5 + mode7) + ORFEES, each with its own limits; lanes on top
# for df in (df_mode5_new, df_mode7_new):
#     ax[0].pcolormesh(df.index, df.columns, df.values.T, cmap="RdYlBu_r",
#                      vmin=df_limits(df)[0], vmax=df_limits(df)[1])
# ax[0].pcolormesh(times, freqs, data, cmap="RdYlBu_r",
#                  vmin=spec_limits(orfees_spec_i)[0],
#                  vmax=spec_limits(orfees_spec_i)[1])

# ax[0].plot(t0 + pd.to_timedelta(tg, "s"), fU, color="lime",    lw=1.5, label="UF fit")
# ax[0].plot(t0 + pd.to_timedelta(tg, "s"), fL, color="magenta", lw=1.5, label="LF fit")
# ax[0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
# ax[0].set_xlim(pd.Timestamp("2025-10-06 08:57"), pd.Timestamp("2025-10-06 09:02"))
# ax[0].set_ylim(290, 120)                       # high freq at bottom; spans both instruments
# ax[0].set_xlabel("Time (UT)"); ax[0].set_ylabel("Frequency (MHz)")
# ax[0].set_title("Band-split lanes (I-LOFAR + ORFEES)"); ax[0].legend(fontsize=8)

# # (b) height-time and shock speed for ref_model, both harmonics
# ax2 = ax[1]; axb = ax2.twinx()
# for s, st in harm_style.items():
#     d = results[(ref_model, s)]
#     ax2.plot(tg, d["r"],    st["ls"], color="firebrick", label=f"height ({st['tag']})")
#     axb.plot(tg, d["v_sh"], st["ls"], color="steelblue", label=f"$v_{{sh}}$ ({st['tag']})")
# ax2.set_xlabel("time since onset [s]")
# ax2.set_ylabel(r"$r\,/\,R_\odot$", color="firebrick")
# axb.set_ylabel(r"$v_{\rm sh}$ [km/s]", color="steelblue")
# ax2.set_title(f"Height-time  ({ref_model})")
# ax2.legend(loc="upper left", fontsize=8)

# # (c) B(r) for all models, both harmonics: colour = model, line style = F/H
# colors = plt.cm.viridis(np.linspace(0, 0.85, len(models)))
# ax3 = ax[2]
# for (name, _), c in zip(models, colors):
#     for s, st in harm_style.items():
#         dd = results[(name, s)]
#         if np.isfinite(dd["r"]).any():
#             ax3.plot(dd["r"], dd["B"], st["ls"], color=c, lw=1.4)
# rr = np.linspace(1.05, 2.0, 100)
# mag_field = 0.5 * (rr - 1.0) ** (-1.5)
# ax3.plot(rr, mag_field, "k:", lw=1)        # Dulk-McLean 1978
# ax3.set_xlabel(r"$r\,/\,R_\odot$"); ax3.set_ylabel(r"$B$ [G]")
# ax3.set_yscale("log"); ax3.set_title("Coronal magnetic field")

# from matplotlib.lines import Line2D
# model_handles = [Line2D([], [], color=c, lw=2, label=n)
#                  for (n, _), c in zip(models, colors)]
# band_handles = [Line2D([], [], color="0.3", ls="-",  label="fundamental"),
#                 Line2D([], [], color="0.3", ls="--", label="harmonic"),
#                 Line2D([], [], color="k",   ls=":",  label="Dulk-McLean 78")]
# leg1 = ax3.legend(handles=model_handles, fontsize=7, loc="upper right")
# ax3.add_artist(leg1)
# ax3.legend(handles=band_handles, fontsize=7, loc="lower left")

# fig.tight_layout()
# plt.show()